# B03：正则化与分类入门（辅修学生练习）

2 实验学时，T06 后。两段合在一个 Notebook，先比较 3 个岭候选，再观察分类概率。数据、路径、预处理和绘图已提供。

只有训练/验证，无独立最终测试。不改划分，不增加候选。完成 3 个 TODO 后重启全运行，保存到非保护盘并另留副本。

## A1. 回归数据与训练内标准化（已提供）

12 列是不同量级的抽象模拟测量，均假设预测时已知。前两列参与数据生成，其余为噪声特征；不代表真实因果发现。这里保留全部 12 列，用于观察正则化在包含无关输入时的作用；真实任务通常不知道哪些输入参与了生成过程。输入 32 训练、48 验证。

先按固定标记划分，再 fit 标准化；验证只 transform。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_absolute_error, confusion_matrix

candidates = []
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates.extend([parent / "data", parent / "units/regularization/data"])
data_dir = next(
    (
        p
        for p in candidates
        if (p / "regression.csv").is_file() and (p / "classification.csv").is_file()
    ),
    None,
)
if data_dir is None:
    raise FileNotFoundError("保留单元目录，从课程目录启动Jupyter")
reg = pd.read_csv(data_dir / "regression.csv")
features = [f"f{i:02d}" for i in range(1, 13)]
train = reg[reg.split == "train"]
val = reg[reg.split == "validation"]
scaler = StandardScaler().fit(train[features])
X_train = scaler.transform(train[features])
X_val = scaler.transform(val[features])
y_train = train.target.to_numpy()
y_val = val.target.to_numpy()
assert X_train.shape == (32, 12) and X_val.shape == (48, 12)
assert set(train.sample_id).isdisjoint(set(val.sample_id))
np.testing.assert_allclose(scaler.mean_, train[features].mean().to_numpy())
display(reg.head())
print("训练/验证：", X_train.shape, X_val.shape)

## A2. 比较三个预设强度

理论 J=半均方误差＋λ×系数平方和/2；偏置不惩罚。库 Ridge 采用误差平方和＋alpha×系数平方和，所以本次用 **alpha=n_train×λ**。不要直接把λ原样当 alpha。

固定λ=[0,0.1,1]。不进行交叉验证或额外搜索。

完成 fit_ridge 中的模型创建和拟合，后续指标与范数计算已经提供。

In [ ]:
def fit_ridge(lam, X_train, y_train):
    # TODO：补全alpha和fit参数；删去停止提示。
    raise NotImplementedError("请完成拟合")
    model = Ridge(alpha=...)
    model.fit(..., ...)
    return model

In [ ]:
def evaluate_ridge(lam, X_train, y_train, X_val, y_val):
    model = fit_ridge(lam, X_train, y_train)
    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)
    half_mse = np.mean((train_pred - y_train) ** 2) / 2
    norm2 = float(model.coef_ @ model.coef_)
    penalty = lam * norm2 / 2
    return {
        "lambda": lam,
        "alpha": model.alpha,
        "train_MAE": mean_absolute_error(y_train, train_pred),
        "validation_MAE": mean_absolute_error(y_val, val_pred),
        "coef_norm": np.sqrt(norm2),
        "half_MSE": half_mse,
        "penalty": penalty,
        "objective": half_mse + penalty,
    }

In [ ]:
# 教师提供：用独立小样本检查实现，不参与本次候选选择。
def check_ridge_implementation():
    check_X = np.array([[-2.0, 1.0], [-1.0, -1.0], [1.0, 2.0], [3.0, 0.0]])
    check_y = np.array([7.0, 8.0, 13.0, 16.0])
    check_X_val = np.array([[0.0, 4.0], [2.0, -3.0]])
    check_y_val = np.array([25.0, -5.0])
    for check_lam in [0.0, 0.4]:
        reference = Ridge(alpha=len(check_y) * check_lam).fit(check_X, check_y)
        actual = evaluate_ridge(check_lam, check_X, check_y, check_X_val, check_y_val)
        expected_half_mse = np.mean((reference.predict(check_X) - check_y) ** 2) / 2
        expected_norm2 = np.sum(reference.coef_**2)
        expected_penalty = check_lam * expected_norm2 / 2
        expected = {
            "alpha": reference.alpha,
            "half_MSE": expected_half_mse,
            "penalty": expected_penalty,
            "objective": expected_half_mse + expected_penalty,
            "coef_norm": np.sqrt(expected_norm2),
            "train_MAE": mean_absolute_error(check_y, reference.predict(check_X)),
            "validation_MAE": mean_absolute_error(
                check_y_val, reference.predict(check_X_val)
            ),
        }
        for key, value in expected.items():
            np.testing.assert_allclose(
                actual[key],
                value,
                rtol=1e-8,
                atol=1e-10,
                err_msg=f"{key} 不符合训练目标，请核对强度、数据范围与惩罚公式",
            )


check_ridge_implementation()
print("强度映射、训练损失与不含截距的平方惩罚检查通过")

In [ ]:
ridge_results = pd.DataFrame(
    [evaluate_ridge(lam, X_train, y_train, X_val, y_val) for lam in [0.0, 0.1, 1.0]]
)
display(ridge_results.round(3))
assert ridge_results["lambda"].tolist() == [0.0, 0.1, 1.0]
np.testing.assert_allclose(ridge_results.alpha, 32 * ridge_results["lambda"])
np.testing.assert_allclose(
    ridge_results.objective, ridge_results.half_MSE + ridge_results.penalty
)
assert np.isfinite(ridge_results.to_numpy()).all()
selected = ridge_results.loc[ridge_results.validation_MAE.idxmin(), "lambda"]
print("本次验证MAE选择的lambda：", selected)
fig, ax = plt.subplots(figsize=(6, 3.2))
ridge_results.set_index("lambda")[["train_MAE", "validation_MAE"]].plot.bar(
    ax=ax, rot=0
)
ax.set_ylabel("MAE")
fig.tight_layout()
plt.show()

## B1. 分类模型（完整提供）

分类使用独立提供的合成数据与训练代码。C 固定 1.0，控制正则化的方向与λ相反；不把 C 数值与岭的λ等同，不调 C。训练内标准化，模型只 fit 训练数据。

In [ ]:
cls = pd.read_csv(data_dir / "classification.csv")
c_features = ["planned_load_ratio", "ambient_index"]
c_train = cls[cls.split == "train"]
c_val = cls[cls.split == "validation"]
c_scaler = StandardScaler().fit(c_train[c_features])
CX_train = c_scaler.transform(c_train[c_features])
CX_val = c_scaler.transform(c_val[c_features])
cy_train = c_train.over_limit.to_numpy()
cy_val = c_val.over_limit.to_numpy()
classifier = LogisticRegression(C=1.0, max_iter=1000).fit(CX_train, cy_train)
print("概率列对应类别：", classifier.classes_)
print("训练/验证：", CX_train.shape, CX_val.shape)

## B2. 提取正类概率并决定类别

`predict_proba` 返回每类的概率列。先从 classes_找出标签 1 的列索引，再得到一维 p_positive。不要将类别预测当作概率。

In [ ]:
# TODO：选择正类1对应的概率列；完成后删除停止提示。
raise NotImplementedError("请提取正类概率")
positive_index = list(classifier.classes_).index(1)
all_probabilities = classifier.predict_proba(CX_val)
p_positive = all_probabilities[:, ...]

In [ ]:
def decide(p, threshold):
    # TODO：补全比较表达式；完成后删除停止提示。
    raise NotImplementedError("请完成阈值判断")
    return (...).astype(int)

In [ ]:
expected_positive_index = list(classifier.classes_).index(1)
expected_positive = classifier.predict_proba(CX_val)[:, expected_positive_index]
np.testing.assert_allclose(
    p_positive,
    expected_positive,
    rtol=1e-10,
    atol=1e-12,
    err_msg="请提取 classes_ 中标签 1 对应的概率列，不能使用类别预测或负类概率",
)
assert p_positive.shape == (40,)
assert ((p_positive >= 0) & (p_positive <= 1)).all()
np.testing.assert_array_equal(decide([0.2, 0.3, 0.8], 0.3), [0, 1, 1])
classification_rows = []
for threshold in [0.5, 0.3]:
    pred = decide(p_positive, threshold)
    tn, fp, fn, tp = confusion_matrix(cy_val, pred, labels=[0, 1]).ravel()
    classification_rows.append(
        {"threshold": threshold, "TP": tp, "FP": fp, "FN": fn, "TN": tn}
    )
classification_results = pd.DataFrame(classification_rows)
display(classification_results)
display(
    pd.DataFrame(
        {
            "id": c_val.batch_id.to_numpy(),
            "y": cy_val,
            "p": p_positive,
            "pred_05": decide(p_positive, 0.5),
            "pred_03": decide(p_positive, 0.3),
        }
    ).head(8)
)

## C. 三项解释与提交

1. 哪个λ的验证 MAE 最小？是否也是训练 MAE 最小或系数范数最小？引用表中数值，说明不能只按其中一项选。
2. 标准化参数来自哪里？若改用厘米等单位却不处理尺度，对惩罚有什么影响？
3. 分类阈值下降时 FP/FN 如何变化？模型概率是否改变？为什么不能把当前结果称为独立最终测试？

**我的回答：**（写在此处）

重启内核全运行，保存一个 Notebook 及输出到非保护盘并另留副本，不交重复长报告。